In [ ]:
import os
os.chdir('???')
os.getcwd()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import datetime
%matplotlib inline

In [ ]:
orig_df = pd.read_csv('airline_passengers.csv')
orig_df.head()

In [ ]:
orig_df.dtypes

In [ ]:
df = orig_df.copy()

### Handling Missing Times and Missing Values

In [ ]:
# find missing value percent for each variable
null_percent = df.isnull().sum()/len(df)*100
null_percent

In [ ]:
# Frequenct String: https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#offset-aliases
# Frequency examples: 'YE' = year, 'QE' = quarter, 'W' = week, 'D' = day, 'h' = hour, 'min' = minute, 's' = second
# >>> forward filling
# df[value_var] = df[value_var].asfreq(freq='YE', method='ffill')     
# >>> backward filling
# df[value_var] = df[value_var].asfreq(freq='YE', method='bfill')    
# >>> resample (average)
# df[value_var] = df[value_var].resample('YE').mean()  

### Set time index

#### String Time Format Code List: https://strftime.org/

In [ ]:
df['Month'] = pd.to_datetime(df['Month'], format='%Y-%m') # Match format according to the data pattern
df.head()

In [ ]:
df.dtypes

In [ ]:
# Set column with datetime as index of dataframe
df = df.set_index('Month')
df.head()

In [ ]:
df.index

In [ ]:
# Visualization timeseries
df.plot()
plt.show()

### Data Preprocessing

In [ ]:
### Detecting and removing outliers using SD

In [ ]:
from scipy.stats import zscore
z_scores = zscore(df)
abs_z_scores = np.abs(z_scores)
abs_z_scores[1:10]

In [ ]:
zscore_threshold = 3 #any data above mean +/- (this_threshold)*sd is considered outliers
outliers = (abs_z_scores > zscore_threshold)  # Z-score threshold
df_outlier_removed = df[~outliers]   
df_outlier_removed[1:10]

In [ ]:
# Plot to compare original and outlier_removed
fig, ax = plt.subplots(2,1)
temp_line, = ax[0].plot(df.index, df, label='original', color='r')
temp_line, = ax[1].plot(df_outlier_removed.index, df_outlier_removed, label='outliers removed', color='m')
ax[0].legend(loc='upper left')
ax[1].legend(loc='upper left')
plt.show()

In [ ]:
# See effect of outlier removal by changing zscore_threshold value
plt.clf()
zscore_threshold = 1.5 #any data above mean +/- (this_threshold)*sd is considered outliers
outliers = (abs_z_scores > zscore_threshold)  # Z-score threshold
df_outliers = df[outliers]
df_nonoutliers = df[~outliers]

fig, ax = plt.subplots(3,1)
temp_line, = ax[0].plot(df.index, df, label='original', color='r')
temp_line, = ax[1].plot(df_outliers.index, df_outliers, label='outliers', color='m')
temp_line, = ax[2].plot(df_nonoutliers.index, df_nonoutliers, label='non-outliers', color='c')
ax[0].legend(loc='upper left')
ax[1].legend(loc='upper left')
ax[2].legend(loc='upper left')
ax[0].set_ylim([100, 700])
ax[1].set_ylim([100, 700])
ax[2].set_ylim([100, 700])

print(df.shape)
print(df_nonoutliers.shape)
plt.show()

In [ ]:
### Detecting and removing outliers using percentile

In [ ]:
from scipy.stats.mstats import winsorize

# Limits outliers to the 5th and 95th percentiles
df_outlier_removed_p = winsorize(df['Thousands of Passengers'], limits=[0.05, 0.05])

# Plot to compare original and outlier_removed
fig, ax = plt.subplots(2,1)
temp_line, = ax[0].plot(df.index, df, label='original', color='r')
temp_line, = ax[1].plot(df.index, df_outlier_removed_p, label='outliers removed by percentile', color='m')
ax[0].legend(loc='upper left')
ax[1].legend(loc='upper left')
plt.show()

### Process Timeseries

### Frequency Setting

#### Frequenct String: https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#offset-aliases

#### Frequency examples: 'YE' = year, 'QE' = quarter, 'W' = week, 'D' = day, 'h' = hour, 'min' = minute, 's' = second

In [ ]:
# To check whether the frequency of timeseries is correct or not
# Frequency strings can be looked at: https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#offset-aliases
df.index.inferred_freq

In [ ]:
# Sampling using resample and mean
sampling_set = df['Thousands of Passengers'].resample('YE').mean()  # one sample = average over time 
sampling_set

In [ ]:
# Sampling using asfreq (ffill = forward fill, bfill = backward fill)
ffill_freq_set = df['Thousands of Passengers'].asfreq(freq='YE', method='ffill')     # one sample = data selected at end of time
ffill_freq_set

In [ ]:
# Sampling using asfreq (ffill = forward fill, bfill = backward fill)
bfill_freq_set = df['Thousands of Passengers'].asfreq(freq='YE', method='bfill')     # one sample = data selected at beginning of time
bfill_freq_set

In [ ]:
# Plot sample sets
plt.clf()
fig = plt.figure(figsize=(16,6)) 
plt.plot(df['Thousands of Passengers'].index, df['Thousands of Passengers'])              
plt.plot(sampling_set.index, sampling_set, ':')              
plt.plot(ffill_freq_set.index, ffill_freq_set, '--')              
plt.plot(bfill_freq_set.index, bfill_freq_set, '-.')              
plt.legend(['original','resample','asfreq_ffill','asfreq_bfill'], loc='upper left')
plt.show()

### Time shifting

In [ ]:
# Before shifting
df.head()

In [ ]:
# Shifting 1 unit
shifted_period = 1
df_shifted = df.shift(periods=shifted_period)
df_shifted.head() 

In [ ]:
plt.clf()
fig = plt.figure(figsize=(16,6)) 
plt.plot(df['Thousands of Passengers'].index, df['Thousands of Passengers'])              
plt.plot(df_shifted.index, df_shifted['Thousands of Passengers'], ':')              
plt.legend(['Original','Shifted'],loc='upper left')
plt.show()

## Smoothing: Moving Average

In [ ]:
# one-sided moving average, window size = 6
df['6-month-SMA'] = df['Thousands of Passengers'].rolling(window=6).mean()

In [ ]:
# one-sided moving average, window size = 12
df['12-month-SMA'] = df['Thousands of Passengers'].rolling(window=12).mean()

In [ ]:
# two-sided moving average, window size = 12
df['6-month-SMA-Center'] = df['Thousands of Passengers'].rolling(window=6,center=True).mean()

In [ ]:
df.head(15)

In [ ]:
df.tail(15)

In [ ]:
plt.clf()
df.plot(figsize=(10,8))
plt.show()

## Smoothing: Exponential Smoothing

In [ ]:
df['EWMA-0.3'] = df['Thousands of Passengers'].ewm(alpha=0.3,adjust=False).mean()

In [ ]:
df['EWMA-0.6'] = df['Thousands of Passengers'].ewm(alpha=0.6,adjust=False).mean()

In [ ]:
df[['Thousands of Passengers','EWMA-0.3','EWMA-0.6']].plot(figsize=(10,8))
plt.show()

## Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
result = seasonal_decompose(df['Thousands of Passengers'])
fig = plt.figure(figsize=(15,8))  
result.plot()
plt.show()

In [ ]:
result.trend.plot()
plt.show()

In [ ]:
fig = plt.figure(figsize=(15,8))  
result.seasonal[1:100].plot()
plt.show()

In [ ]:
result.resid.plot()
plt.show()